# 03 — What funds hold

**The question:** *Which Brazilian funds own PETR4, how much of it, and who is
holding Petrobras' debentures?*

This is the one edge in the warehouse that joins the fund universe to the quote
tape. CVM's CDA filing publishes the **B3 ticker** a fund holds (block 4), the
**CNPJ** of a held fund (block 2) and the **issuer's own CNPJ** for a debenture
(block 6). Nothing here is matched by name — the identifier arrives on the
filed row or the row does not exist.

Endpoints: `fund_holdings`, `fund_debentures`.

In [ ]:
# The SDK is not on PyPI. From the repository root:
#
#     pip install -e sdk/
#
# Auth is the shared publishable key printed in the docs. It is for TESTING:
# everyone reading the docs has the same one, so it identifies the project and
# not you. It puts you on the ANONYMOUS tier. Set SILO_TOKEN to a GitHub
# sign-in token (notebook 00) to run signed in.
import os

os.environ.setdefault("SILO_URL", "https://zcjbtpxuhdekpwcxmepn.supabase.co")
os.environ.setdefault(
    "SILO_ANON_KEY", "sb_publishable__yfFQsykAglrvc9GS6_PYw_B24ex437"
)

import pandas as pd

from silo_client import SiloClient

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

silo = SiloClient()
print(f"tier            : {silo.tier}")
print(f"catalog version : {silo.catalog()['version']}")

In [ ]:
def as_of(*datasets: str) -> pd.DataFrame:
    """Print how fresh every dataset this notebook relies on actually is.

    Run this FIRST, every time. A stale warehouse then shows up in the output
    instead of being silently baked into a number further down.

      as_of            the newest period that has landed AND has elapsed.
                       This is freshness.
      complete_through the newest period classified COMPLETE — what the
                       default windows serve.
      newest_period    the newest period KEY. It can sit in the FUTURE when a
                       family files forward-dated (FIP is keyed 31-December).
                       Never read this as freshness.
      landed_at        when ingest last SUCCEEDED. A later failed run never
                       advances it.
      notes            a caveat the dates cannot carry. Printed in full below,
                       never summarised, never dropped.
    """
    cov = pd.DataFrame(silo.coverage())
    rows = cov[cov["dataset"].isin(datasets)].copy()
    missing = set(datasets) - set(rows["dataset"])
    if missing:
        raise RuntimeError(f"coverage() has no row for {sorted(missing)}")
    print(
        rows[
            ["dataset", "as_of", "complete_through", "newest_period", "landed_at"]
        ].to_string(index=False)
    )
    for r in rows.itertuples():
        if r.notes:
            print(f"\n  CAVEAT [{r.dataset}]\n  {r.notes}")
    return rows.set_index("dataset")

In [ ]:
COVERAGE = as_of("funds", "fund_nav", "quotes")

Holdings ride on the same CVM filing calendar as `fund_nav`, so
`complete_through` on the fund rows is the honest edge of the window below.

## Who holds PETR4?

`fund_holdings` runs in both directions and you pick one:

```python
silo.fund_holdings(ticker="PETR4")          # which funds hold it
silo.fund_holdings(cnpj="...")              # what this fund holds
silo.fund_holdings(cnpj="...", kind="fund") # held FUNDS, not shares
```

Asking for both, or neither, is an error rather than a silently narrowed or
unbounded scan.

In [ ]:
TICKER = "PETR4"
PERIOD = "2026-06-01"          # one month, keyed on the first of the month

holders = pd.DataFrame(
    silo.fund_holdings(ticker=TICKER, start=PERIOD, end=PERIOD, limit=500)
)
print(f"{len(holders):,} holding rows for {TICKER} in {PERIOD[:7]}")
print(f"(anonymous rows ceiling is {silo.limits()['tiers']['anon']['fund_holdings_rows']};"
      f" a full month can exceed it — narrow or sign in)")
holders.head()

### Rows are as filed, and are never summed across them

Look at `tp_aplic` (application type) and `tp_negoc` (trading intent). CVM
publishes **one row per (fund, month, security, application type, trading
intent)** and this API serves them exactly that way.

Collapsing them is the mistake the holdings key audit exists to prevent: the
same fund can hold the same ticker under two intents, and adding those two rows
is right for "total position" and wrong for almost everything else. Decide which
you want, in your notebook, visibly.

In [ ]:
print(holders.groupby(["tp_aplic", "tp_negoc"]).size().to_frame("rows").to_string())
print()
dupes = (holders.groupby(["cnpj", "held_id"]).size()
         .loc[lambda s: s > 1])
print(f"funds filing more than one row for {TICKER} this month: {len(dupes)}")
if len(dupes):
    print(dupes.head().to_string())

### The largest holders

`vl_merc_pos_final` is the position's **market value as the fund reported it**,
not a recomputation from the quote tape. It is one row's value; summing to a
per-fund total is a deliberate choice, so it is made here rather than upstream.

In [ ]:
by_fund = (holders.groupby("cnpj", as_index=False)
           .agg(rows=("held_id", "size"),
                quantity=("qt_pos_final", "sum"),
                market_value=("vl_merc_pos_final", "sum"))
           .sort_values("market_value", ascending=False)
           .head(10))

# Names come from the registry, by CNPJ — never from a name match.
names = {f["cnpj"]: f["fund_name"] for f in silo.view(
    "funds", cnpj=f"in.({','.join(by_fund['cnpj'])})",
    select="cnpj,fund_name", limit=50)}
by_fund["fund_name"] = by_fund["cnpj"].map(names).str.slice(0, 52)

print(f"Top holders of {TICKER}, {PERIOD[:7]} — as filed, summed across "
      f"(tp_aplic, tp_negoc) here in the notebook\n")
print(by_fund[["cnpj", "fund_name", "quantity", "market_value"]]
      .to_string(index=False, float_format=lambda v: f"{v:,.0f}"))

### Sanity, not validation

The filed market value and the tape's close are two independent observations. A
close comparison is a useful smell test — it is **not** a correction, and
neither number is adjusted to match the other.

In [ ]:
close = silo.quote_latest(TICKER)[0]["close"]
implied = by_fund["market_value"] / by_fund["quantity"]

print(f"latest {TICKER} close (unadjusted, "
      f"{COVERAGE.loc['quotes', 'as_of']}): R$ {close:,.2f}")
print(f"implied price from the {PERIOD[:7]} filings: "
      f"R$ {implied.min():,.2f} .. R$ {implied.max():,.2f}")
print()
print("These are months apart and neither is adjusted for corporate actions.")
print("A difference is a difference, not an error to reconcile away.")

## The other direction: what one fund holds

In [ ]:
BIG = by_fund.iloc[0]["cnpj"]
print(names.get(BIG, BIG), "\n")

port = pd.DataFrame(silo.fund_holdings(cnpj=BIG, start=PERIOD, end=PERIOD,
                                       limit=500))
if port.empty:
    print("no equity block filed for this fund in this month")
else:
    top = (port.groupby("held_id", as_index=False)
           .agg(market_value=("vl_merc_pos_final", "sum"))
           .sort_values("market_value", ascending=False).head(12))
    top["share_of_listed_book_pct"] = (
        100 * top["market_value"] / port["vl_merc_pos_final"].sum()).round(2)
    print(top.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))
    print()
    print("CAVEAT: the denominator is the fund's LISTED EQUITY block only")
    print("(CDA block 4). It is not the fund's NAV, and these percentages")
    print("do not sum to the whole portfolio.")

### `kind="fund"` — funds of funds

Block 2 carries the **CNPJ of the held fund**, plus CVM's own published
`emissor_ligado` flag (is the issuer related to the administrator?). That flag
is filed, not inferred.

In [ ]:
held_funds = pd.DataFrame(
    silo.fund_holdings(cnpj=BIG, start=PERIOD, end=PERIOD, kind="fund", limit=200)
)
if held_funds.empty:
    print(f"{BIG} filed no block-2 (fund quota) holdings in {PERIOD[:7]}")
else:
    print(held_funds[["held_id", "held_name", "emissor_ligado",
                      "vl_merc_pos_final"]].head(10)
          .to_string(index=False, float_format=lambda v: f"{v:,.0f}"))

## Debentures are a different shape

`fund_debentures` is CDA **block 6**, and it is deliberately not a third `kind`
of `fund_holdings`: a debenture's identity is `(issuer, maturity, rate
structure, application type)`, and those columns have nowhere to go in the
equity shape.

Two consequences:

* **Two series of one issuer maturing on the same day at different coupons are
  different securities.** Rows are never summed across them.
* **The issuer is its own filed CPF/CNPJ.** `p_issuer` also takes a listed
  company's ticker or CVM code, resolved *only* through CVM's published FCA map.
  Most debenture issuers are not listed at all, and for those the CNPJ is the
  only way in — `issuer_tickers` comes back `None`, which means "not listed",
  not "unknown".

In [ ]:
deb = pd.DataFrame(silo.fund_debentures(issuer=TICKER, start="2026-05-01",
                                        end=PERIOD, limit=500))
print(f"{len(deb):,} rows: funds holding paper issued by {TICKER}'s company\n")
deb[["cnpj", "period", "issuer_id", "tp_ativo", "maturity", "indexer",
     "indexer_pct", "coupon_pct", "vl_merc_pos_final"]].head(6)

### One issuer, one maturity, several securities

`issuer_tickers` is a **list** — CVM's published FCA map can carry several
active codes for one company, and this row is the issuer's own CNPJ, not a
name match. Note `PETR*` alongside `PETR3` and `PETR4`: the map is served as
published, wildcards and all.

Then look at what shares a `maturity`. The same issuer, the same maturity date,
and rows that are nonetheless **different securities** — a debenture and a
commercial paper, on different application types. Summing them gives you
"exposure to the issuer", which is a real question; it is just not the same
question as "how much of this bond is held".

In [ ]:
identity = deb[["issuer_id", "issuer_kind", "issuer_tickers"]].copy()
identity["issuer_tickers"] = identity["issuer_tickers"].map(
    lambda v: ", ".join(v) if isinstance(v, list) else v)
print("issuer identity, as filed and as published by the FCA map:")
print(identity.drop_duplicates().to_string(index=False))
print()
print("(issuer_tickers is None when the issuer is NOT LISTED — which is most")
print(" debenture issuers. None means 'not listed', never 'unknown'.)")
print()

shape = (deb.groupby(["maturity", "tp_ativo", "indexer", "indexer_pct",
                      "coupon_pct"], dropna=False)
         .agg(funds=("cnpj", "nunique"),
              market_value=("vl_merc_pos_final", "sum"))
         .reset_index())
print(f"distinct (maturity, security type, rate structure) combinations: "
      f"{len(shape)}")
print(shape.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

## Where this goes next

* Notebook `04` uses the same *as filed, never summed* discipline on a FIDC's
  receivables book, where the hierarchy makes double-counting easy.
* Notebook `05` follows the issuer the other way: from a ticker to the
  company's filed financial statements.

---

## The rules this notebook obeyed

* **Nothing was filled.** No forward-fill, no interpolation, no carried-forward
  last observation. A gap in a chart is a gap in the filings.
* **Every caveat was printed beside its number** — `coverage().notes`,
  `catalog().regime_breaks`, `catalog().applicability`, `float_basis` — rather
  than left in a docstring somewhere.
* **Freshness came from `coverage()`**, called before anything was claimed.

The contract these rules come from is
[Conventions & limits](https://octo-98895abd.mintlify.site/api-docs/conventions),
and its machine-readable twin is `POST /rpc/catalog`.